In [1]:
#defining a zero-lag ema
def zlema(series, period):
    ema1 = talib.EMA(series, period)
    ema2 = talib.EMA(ema1, period)
    return 2 * ema1 - ema2
#implementig vectorized operation to define crossovers 
def vectorized_crossover(series1, series2):
    return (series1 > series2) & (series1.shift(1) < series2.shift(1))
    
def vectorized_crossunder(series1, series2):
    return (series1 < series2) & (series1.shift(1) > series2.shift(1))
    
#defining a zero-lag macd, cond_buy is defined whenever macd line and signal line crosses under zero line
#simmetrically for cond_sell
def macd_impl(df):
    df['fast_period'] = zlema(df['Close'], 12)
    df['slow_period'] = zlema(df['Close'], 26)
    df['macd'] = df['fast_period'] - df['slow_period']
    df['signal'] = zlema(df['macd'], 9)
    df['hist'] = df['macd'] - df['signal']
    df['atr'] = talib.ATR(df['High'], df['Low'], df['Close'], timeperiod=14)
    
    cond_buy = (
    vectorized_crossover(df['macd'], df['signal']) & 
    (df['macd'] < 0)
    )
    
    cond_sell = (
    vectorized_crossunder(df['macd'], df['signal']) & 
    (df['macd'] > 0)
    )
    conditions = [cond_buy, cond_sell]
    choices    = [1, -1]
#Trade_Direction will translate cond_buy and cond_sell into 1 and -1
    df['Trade_Direction'] = np.select(conditions, choices, default=0)
    df['stop_loss'] = np.where(
        df['Trade_Direction'] > 0,
        df['Close'] - (df['atr'] * 2.5),
        df['Close'] + (df['atr'] * 2.5)
         )
    #df['Trade_Direction'] = df['Trade_Direction'].shift(-1)
    return df

In [2]:
#this function will aling the trade result with the trigger candle
def implement_trades(df, df_results):
    
    df['Trades'] = np.nan
    
    df.iloc[df_results['EntryBar'].values, df.columns.get_loc('Trades')] = df_results['PnL'].values
    
    df.loc[df['Trades'] > 0, 'Trades'] = 1  #PnL > 0 win trade
    df.loc[df['Trades'] < 0, 'Trades'] = 0 #PnL <0 loss trade
    df['Trades'] = df['Trades']
    return df

In [3]:

def detect_highs_lows(df):
    
    df['recent_high_fast'] = talib.MAX(df['High'], timeperiod=7)
    df['recent_low_fast'] = talib.MIN(df['Low'].dropna().astype(float), timeperiod=7)
    
    df['recent_high_med'] = talib.MAX(df['High'], timeperiod=21)
    df['recent_low_med'] = talib.MIN(df['Low'].dropna().astype(float), timeperiod=21)
    
    df['recent_high_slow'] = talib.MAX(df['High'], timeperiod=100)
    df['recent_low_slow'] = talib.MIN(df['Low'].dropna().astype(float), timeperiod=100)
    
    df['from_high_fast'] = (df['Close'] - df['recent_high_fast']) / df['recent_high_fast']
    df['from_low_fast'] = (df['Close'] - df['recent_low_fast']) / df['recent_low_fast']
    
    df['from_high_med'] = (df['Close'] - df['recent_high_med']) / df['recent_high_med']
    df['from_low_med'] = (df['Close'] - df['recent_low_med']) / df['recent_low_med']
    
    df['from_high_slow'] = (df['Close'] - df['recent_high_slow']) / df['recent_high_slow']
    df['from_low_slow'] = (df['Close'] - df['recent_low_slow']) / df['recent_low_slow']

    return df

In [4]:
def indicators_features(df):
    
    df['sar'] = talib.SAR(df['High'], df['Low'])
    df['Rsi_9'] = talib.RSI(df['Close'], timeperiod = 9)
    df['Rsi_14'] = talib.RSI(df['Close'], timeperiod = 14)
    
    return df

In [5]:
def ema_features(df):
    
    df['ema_50'] = talib.EMA(df['Close'], 50)
    df['ema_200'] = talib.EMA(df['Close'], 200)
    
    df['ema_50_slope'] = talib.LINEARREG_SLOPE(df['ema_50'], timeperiod = 50)
    df['ema_200_slope'] = talib.LINEARREG_SLOPE(df['ema_200'], timeperiod = 50)
    
    df['ratio_ema_50'] = abs(df['Close'] / df['ema_50'])
    df['ratio_ema_200'] = abs(df['Close'] / df['ema_200'])
    
    df['above_ema_50'] = (df['Close'] > df['ema_50']).astype(int)
    df['above_ema_200'] = (df['Close'] > df['ema_200']).astype(int)

    df['distance_from_ema_50'] = (df['Close'] - df['ema_50']).abs() / df['Close']
    df['distance_from_ema_200'] = (df['Close'] - df['ema_200']).abs() / df['Close']

    return df

In [6]:
def detect_market_regimes(df):
    #here i calc the volatility of the price in the higher tf
    df['volatility_15min'] = df['Close'].pct_change().rolling(3).std()
    df['high_volatility_15min'] = (df['volatility_15min'] > df['volatility_15min'].expanding().quantile(0.7)).astype(int)
    
    df['volatility_1h'] = df['Close'].pct_change().rolling(12).std()
    df['high_volatility'] = (df['volatility_1h'] > df['volatility_1h'].expanding().quantile(0.7)).astype(int)

    df['trend_strenght_4h'] = df['Close'].pct_change(48).abs()
    df['strong_trend'] = (df['trend_strenght_4h'] > df['trend_strenght_4h'].expanding().quantile(0.7)).astype(int)
    #this is the price change in 1h tf standardized over current price
    df['price_change_4h'] = (df['High'].rolling(48).max() - df['Low'].rolling(48).min()) / df['Close']
    df['range_bound'] = (df['price_change_4h'] < df ['price_change_4h'].expanding().quantile(0.3)).astype(int)
    
    return df

In [7]:
def tranding_features(df):
    df['from_high_med_vol_adj'] = df['from_high_med'] / (1 + df['high_volatility'] * 0.5)
    df['from_low_med_vol_adj'] = df['from_low_med'] / (1 + df['high_volatility'] * 0.5)
    
    df['from_high_fast_trend_boost'] = df['from_high_fast'] * (1 + df['strong_trend'] * 0.3)
    df['from_low_fast_trend_boost'] = df['from_low_fast'] * (1 + df['strong_trend'] * 0.3)
    
    df['rsi_range_sensitive'] = df['Rsi_9'] * (1 + df['range_bound'] * 0.4)
    
    df['volatility_confirmed_signal'] = df['from_high_med'] * (1 - df['high_volatility'] * 0.6)
    df['trend_confirmed_signal'] = df['from_low_med'] * (1 + df['strong_trend'] * 0.4)

    return df

In [8]:
def regime_specific_features(df):
    
    df['volatility_breakout'] = df['from_high_fast'] * df['high_volatility']
    df['panic_buy_signal'] = (df['from_low_fast'] > 0.02) & df['high_volatility']
    
    df['range_extreme'] = ((df['from_high_med'] > 0.01) | (df['from_low_med'] < -0.01)) & df['range_bound']
    df['tight_range_bounce'] = (df['from_low_fast'] < -0.005) & df['range_bound']
    
    df['trend_momentum'] = df['ema_50_slope'] * df['strong_trend']
    df['trend_pullback'] = (df['from_low_med'] < -0.01) & df['strong_trend']
    
    return df

In [15]:
import pandas as pd
import talib
import numpy as np
from backtesting import Strategy, Backtest

In [16]:
df = pd.read_csv('btcusdt_spot_last_3years_italy.csv')

In [17]:
df.columns = df.columns.str.capitalize()
macd_impl(df)
detect_highs_lows(df)
indicators_features(df)
detect_market_regimes(df)
ema_features(df)
df.dropna(inplace = True)

In [22]:
class trade_status_strategy(Strategy):
    
    def init(self):
        #empty function in which we should put internal calcs for indicators, required even if empty
        pass

    def next(self):

        price = self.data.Close[-1]
        current_atr = self.data.atr[-1]
        trade_signal = self.data.Trade_Direction[-1]

        if not self.position:
            if trade_signal == -1:  # Short Signal
                # Calculate levels based on the moment of the signal
                sl_price = price + (current_atr * 2.5)
                tp_price = price - (current_atr * 5)
                
                # Execute at current close (or next open) using these static values
                self.sell(size=0.01, sl=sl_price, tp=tp_price)

            elif trade_signal == 1:  # Long Signal
                sl_price = price - (current_atr * 2.5)
                tp_price = price + (current_atr * 5)
                
                self.buy(size=0.01, sl=sl_price, tp=tp_price)

In [24]:
bt = Backtest(
    df,
    trade_status_strategy,
    cash=100000000,
    exclusive_orders=True,
    trade_on_close=True
)
results = bt.run()
results

/home/matteozamaro/venv/lib/python3.11/site-packages/backtesting/backtesting.py:1212: FutureWarning: Index.is_numeric is deprecated. Use pandas.api.types.is_any_real_numeric_dtype instead
  (data.index.is_numeric() and
/tmp/ipykernel_28399/1750268398.py:1: UserWarning: Data index is not datetime. Assuming simple periods, but `pd.DateTimeIndex` is advised.
  bt = Backtest(
/tmp/ipykernel_28399/1750268398.py:8: UserWarning: Some trades remain open at the end of backtest. Use `Backtest(..., finalize_trades=True)` to close them and include them in stats.
  results = bt.run()


Start                                   248.0
End                                  315343.0
Duration                             315095.0
Exposure Time [%]                    92.17318
Equity Final [$]              101546210.59543
Equity Peak [$]               101634813.44511
Return [%]                            1.54621
Buy & Hold Return [%]                436.1063
Return (Ann.) [%]                         0.0
Volatility (Ann.) [%]                     NaN
Sharpe Ratio                              NaN
Sortino Ratio                             NaN
Calmar Ratio                              0.0
Alpha [%]                             1.04792
Beta                                  0.00114
Max. Drawdown [%]                    -0.32218
Avg. Drawdown [%]                    -0.00942
Max. Drawdown Duration                57284.0
Avg. Drawdown Duration              433.30663
# Trades                               5537.0
Win Rate [%]                         34.83836
Best Trade [%]                    

In [25]:
df_results = results['_trades']
implement_trades(df, df_results)
df_trades = df.loc[(df['Trades'] == 0) | (df['Trades'] == 1)]



In [28]:
df.to_csv('btc_3y_with_features.csv')
df_trades.to_csv('btc_trades_no_overlap.csv')